# Parquet: early features, similarity, wedges and prediction

Experimental notebook interface to `extension.pipeline`. The existing implementation remains authoritative. Run cells from top to bottom in a fresh kernel. Outputs are isolated under `output/notebook_extension/`; rerunning replaces that notebook’s outputs.

This is a staged orchestration of existing functions, not an independent reimplementation. Open a function’s source to change its algorithm; substitute a compatible function call below to test a variation. Preserve train-only fitting, splits, matched costs, support thresholds and dependence-aware uncertainty. See the repository methodology before interpreting results.

## 1. Locate the project and import reusable functions
The Python environment must contain the project dependencies plus a Jupyter kernel.

In [ ]:
import os
import sys
from pathlib import Path

# Works when Jupyter starts in this folder or anywhere inside the repository.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "extension/pipeline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the project repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/ds340-notebook-mpl")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/ds340-notebook-cache")
from extension.pipeline import (  # noqa: E402 - repository path must be set before project imports
    SEED,
    Path,
    argparse,
    build_labels,
    figures,
    hashlib,
    importlib,
    json,
    model_labels,
    os,
    pd,
    prepare,
    representations,
    threadpool_limits,
    time,
)

SOURCE_FILE = ROOT / "extension/pipeline.py"

## 2. Configuration
Defaults match the existing CLI, except for isolated paths. Edit this argument list for a separate sensitivity run, using a new output directory. Do not select a model using the held-out results.

In [ ]:
NOTEBOOK_ARGUMENTS = ["--data", "data/kalshi", "--output", "output/notebook_extension/parquet"]

## Parse and validate settings

In [ ]:
parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument("--data", type=Path, default=Path("data/kalshi"))
parser.add_argument("--output", type=Path, default=Path("extension/results"))
parser.add_argument("--max-contracts", type=int, default=60000)
parser.add_argument("--window-hours", type=int, default=24)
parser.add_argument("--min-early-trades", type=int, default=5)
parser.add_argument("--clusters", type=int, default=16)
parser.add_argument("--min-cell-contracts", type=int, default=5)
parser.add_argument("--min-cell-events", type=int, default=3)
parser.add_argument("--min-group-contracts", type=int, default=20)
parser.add_argument("--min-group-events", type=int, default=10)
parser.add_argument("--bootstrap", type=int, default=200)
parser.add_argument("--min-coherence", type=float, default=0.65)
args = parser.parse_args(NOTEBOOK_ARGUMENTS)
if not 0 <= args.min_coherence <= 1:
    parser.error("Coherence must be between zero and one")
if (
    args.bootstrap < 100
    or min(
        args.window_hours,
        args.max_contracts,
        args.min_early_trades,
        args.clusters,
        args.min_cell_contracts,
        args.min_cell_events,
        args.min_group_contracts,
        args.min_group_events,
    )
    <= 0
):
    parser.error("Require positive settings and at least 100 bootstrap repetitions")

# Keep experimental writes away from the established implementation and results.
prototype_root = (ROOT / "output/notebook_extension").resolve()
if prototype_root not in args.output.resolve().parents:
    raise ValueError("Choose an output subdirectory under output/notebook_extension/.")
args.output.mkdir(parents=True, exist_ok=True)
(args.output / "run_manifest.json").write_text(
    json.dumps({"completed": False, "interface": "experimental_notebook"}, indent=2)
)

## Create the isolated run directory

In [ ]:
out = args.output.resolve()
out.mkdir(parents=True, exist_ok=True)

## Record provenance before analysis

In [ ]:
start = time.time()
manifest = {
    "seed": SEED,
    "config": {k: str(v) if isinstance(v, Path) else v for (k, v) in vars(args).items()},
    "versions": {
        p: importlib.metadata.version(p) for p in ["numpy", "pandas", "duckdb", "scipy", "scikit-learn", "matplotlib"]
    },
    "input_files": [
        {"path": str(p), "bytes": p.stat().st_size, "mtime_ns": p.stat().st_mtime_ns}
        for sub in ["markets", "trades"]
        for p in sorted((args.data / sub).glob("*.parquet"))
    ],
    "source_sha256": hashlib.sha256(Path(SOURCE_FILE).read_bytes()).hexdigest(),
    "report_source_sha256": hashlib.sha256(Path(SOURCE_FILE).with_name("report.py").read_bytes()).hexdigest(),
}
manifest["completed"] = False
manifest["notebook_source_sha256"] = hashlib.sha256(
    (ROOT / "notebooks/extension_prototype/01_parquet_prediction.ipynb").read_bytes()
).hexdigest()
manifest["interface"] = "experimental_notebook"
(out / "run_manifest.json").write_text(json.dumps(manifest, indent=2))

## Prepare early-window inputs

In [ ]:
with threadpool_limits(limits=2):
    (contracts, cells) = prepare(args, out)

## Fit training-only text/price representations and save features

In [ ]:
with threadpool_limits(limits=2):
    contracts = representations(contracts, args, out)
    contracts.to_parquet(out / "contract_features.parquet", index=False)
    cells.to_parquet(out / "contract_price_counts.parquet", index=False)

## Construct exact-cost labels and fit held-out models

In [ ]:
with threadpool_limits(limits=2):
    (all_metrics, all_preds, all_coefs) = ([], [], [])
    for approach in ["text", "text_price"]:
        print(f"Estimating {approach} grouped labels...", flush=True)
        labels = build_labels(contracts, cells, approach, args, out)
        print(labels.groupby(["split", "status"]).size(), flush=True)
        (met, pred, coef, rank) = model_labels(labels, approach, args, out)
        all_metrics.append(met)
        all_preds.append(pred)
        all_coefs.append(coef)
        if approach == "text":
            (primary_labels, primary_rank) = (labels, rank)

## Combine metrics and generate figures

In [ ]:
with threadpool_limits(limits=2):
    metrics = pd.concat(all_metrics, ignore_index=True)
    metrics.to_csv(out / "model_comparison.csv", index=False)
    figures(
        primary_labels,
        pd.read_csv(out / "cluster_profiles.csv"),
        metrics,
        pd.concat(all_preds),
        pd.concat(all_coefs),
        primary_rank,
        out,
    )

## Generate the findings report

In [ ]:
from extension.report import write_findings

## Write findings

In [ ]:
write_findings(out)

## Mark the completed run

In [ ]:
manifest["elapsed_seconds"] = time.time() - start
manifest["completed"] = True
(out / "run_manifest.json").write_text(json.dumps(manifest, indent=2))
print(metrics.to_string(index=False), flush=True)
print(f"Completed in {manifest['elapsed_seconds']:.1f}s. Results: {out}", flush=True)

## Inspect results and limitations
A completed manifest means execution finished; it does not validate the research assumptions. This is not a chronological backtest, a causal wording experiment, or an execution-aware trading simulation. Sparse/rejected groups must remain visible. Compare this run with the CLI using identical inputs, settings and software before adopting the notebook route.

In [ ]:
from IPython.display import Image, display

result_dir = args.output.resolve()
print("Results:", result_dir)
print("Completed:", json.loads((result_dir / "run_manifest.json").read_text()).get("completed", False))
for csv_name in ["model_comparison.csv", "coverage.csv", "population_coverage.csv", "heterogeneity_tests.csv"]:
    csv_path = result_dir / csv_name
    if csv_path.exists():
        print(csv_name)
        display(pd.read_csv(csv_path).head(20))
for image_path in sorted(result_dir.glob("*.png")) + sorted((result_dir / "figures").glob("*.png")):
    display(Image(filename=str(image_path)))
report_path = result_dir / "FINDINGS.md"
if not report_path.exists():
    report_path = result_dir / "STRUCTURAL_FINDINGS.md"
if report_path.exists():
    print("Report:", report_path)